# Notebook 06 — RAG + reranking (cross-encoder)

**Prérequis :** exécuter `03` (index FAISS).

**Sortie :** `results/rerank_predictions.json`

**Étape suivante :** `07_function_calling.ipynb`


## 0. Montage Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'


## 1. Installation


In [ ]:
!pip install -q groq sentence-transformers faiss-cpu


## 2. Imports


In [ ]:
# Chemins et constantes reranking
import os, json, time, getpass
import numpy as np
import faiss
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.notebook import tqdm

RAW_PATH       = os.path.join(BASE_PATH, 'data', 'raw')
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')
FAISS_PATH     = os.path.join(BASE_PATH, 'models', 'faiss_index')
os.makedirs(RESULTS_PATH, exist_ok=True)

GROQ_MODEL       = "llama-3.1-8b-instant"
EMBED_MODEL      = "paraphrase-multilingual-MiniLM-L12-v2"
CROSS_MODEL      = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
TOP_M            = 30   # candidats bi-encodeur avant rerank
TOP_K_FINAL      = 5    # chunks après cross-encoder
THROTTLE_S       = 0.5
print("Rerank : bi-encodeur + cross-encoder + Groq")


## 3. Chargement


In [ ]:
def load_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"[ERROR] {path}: {e}")
        return []

print("Chargement FAISS + test...")
test_data   = load_json(os.path.join(PROCESSED_PATH, 'test.json'))
corpus_meta = load_json(os.path.join(FAISS_PATH, 'metadata.json'))
index       = faiss.read_index(os.path.join(FAISS_PATH, 'index.faiss'))
embed_model = SentenceTransformer(EMBED_MODEL)

try:
    reranker = CrossEncoder(CROSS_MODEL)
except Exception as e:
    print(f"[WARN] Reranker indisponible ({CROSS_MODEL}): {e}")
    backup_model = "cross-encoder/ms-marco-MiniLM-L-12-v2"
    print(f"[INFO] Fallback reranker: {backup_model}")
    reranker = CrossEncoder(backup_model)

api_key = getpass.getpass("Clé Groq API : ")
groq_client = Groq(api_key=api_key)
print(f"Index: {index.ntotal} vecteurs | Test: {len(test_data)}")


## 4. Inférence rerank + Groq


In [ ]:
RAG_PROMPT_TEMPLATE = """Tu es un assistant expert. Réponds EN FRANÇAIS, de façon concise et factuelle,
en t'appuyant sur les extraits ci-dessous (ignore le bruit type paywall / navigation).

Contexte :
{context_block}

Question : {question}"""

def retrieve_wide(question, m=TOP_M):
    q_emb = embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, m)
    return [corpus_meta[i] for i in idxs[0] if i < len(corpus_meta)], scores[0].tolist()

def rerank_chunks(question, chunks):
    texts = [c.get('text', c.get('context', ''))[:1200] for c in chunks]
    pairs = [[question, t] for t in texts]
    ce_scores = reranker.predict(pairs, show_progress_bar=False)
    order = np.argsort(-np.array(ce_scores))
    ranked = [chunks[i] for i in order[:TOP_K_FINAL]]
    return ranked, ce_scores.tolist() if hasattr(ce_scores, 'tolist') else list(map(float, ce_scores))

def call_groq_rerank(question, chunks, retries=4):
    context_block = "\n\n".join([
        f"[{i+1} — {c.get('title','')[:60]}] {c.get('text', '')[:900]}"
        for i, c in enumerate(chunks)
    ])
    prompt = RAG_PROMPT_TEMPLATE.format(context_block=context_block, question=question)
    for attempt in range(retries):
        try:
            t0 = time.time()
            r = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
            )
            lat = round((time.time() - t0) * 1000)
            ans = r.choices[0].message.content.strip()
            tin = r.usage.prompt_tokens if r.usage else 0
            tout = r.usage.completion_tokens if r.usage else 0
            return ans, lat, tin, tout
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate_limit' in err.lower():
                time.sleep(5 * (2 ** attempt))
            else:
                time.sleep(1)
    return "", 0, 0, 0

print("Prêt : rerank + Groq")


In [ ]:
rerank_predictions = []
for item in tqdm(test_data, desc="Rerank (FAISS+M+CE -> Groq)"):
    q = item.get('question', '')
    gold = item.get('answer', '')
    chunks, _ = retrieve_wide(q, TOP_M)
    top_chunks, _ = rerank_chunks(q, chunks)
    pred, lat, tin, tout = call_groq_rerank(q, top_chunks)
    rerank_predictions.append({
        "pair_id": item.get('pair_id', ''),
        "question": q,
        "predicted_answer": pred,
        "true_answer": gold,
        "latency_ms": lat,
        "tokens_in": tin,
        "tokens_out": tout,
        "method": "rerank",
        "dataset_type": item.get("dataset_type", ""),
        "question_type": item.get("question_type", ""),
    })
    time.sleep(THROTTLE_S)

outp = os.path.join(RESULTS_PATH, 'rerank_predictions.json')
with open(outp, 'w', encoding='utf-8') as f:
    json.dump(rerank_predictions, f, ensure_ascii=False, indent=2)
print(f"Sauvegardé : {outp} ({len(rerank_predictions)} lignes)")
